# Project 1: Word Embeddings / Recurrent Neural Networks

## Introduction

This project is part of the NLP module held in the spring of 2026. Two classification
architectures are trained and compared on a physical commonsense reasoning task:

- **Word embedding classifier**: fastText embeddings (300-dim, `cc.en.300.bin`) with a
  2-layer feedforward classifier (Linear → ReLU → Dropout → Linear).
- **RNN classifier**: A 2-layer LSTM encoder (PyTorch built-in) whose final hidden state
  is fed into the same 2-layer feedforward classifier.

**Dataset**  
"PIQA: Reasoning about Physical Commonsense in Natural Language" — a binary choice task
where a model selects the more physically plausible solution to a given goal.  
Source: [https://arxiv.org/abs/1911.11641](https://arxiv.org/abs/1911.11641)

**Tools**  
Claude AI was used for the following tasks:
- Helping formulate and clarify reasoning
- General coding assistance

**Weights & Biases**  
All experimental runs are logged and published in the
[W&B report](https://wandb.ai/sacha-vogel-hochschule-luzern/nlp-project1-piqa/reports/NLP-Project-1-PIQA-Dataset--VmlldzoxNjQzMjY4MA).

**Notebook structure**
1. Introduction
2. Setup
3. Preprocessing
4. Model
5. Training
6. Evaluation
7. Interpretation


## Setup

In [1]:
!pip install \
    datasets==4.8.4 \
    fasttext==0.9.3 \
    nltk==3.9.4 \
    numpy==2.4.4 \
    scikit-learn==1.8.0 \
    torch==2.11.0 \
    wandb==0.25.1


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report
from torch.nn.utils.rnn import pack_padded_sequence
import numpy as np
import re
import string
import ssl
import wandb
import nltk
import torch
import torch.nn as nn
import fasttext.util

In [3]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
generator = torch.Generator()
generator.manual_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [4]:
wandb_project = "nlp-project1-piqa"
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/sachavogel/.netrc.
wandb: Currently logged in as: sacha-vogel (sacha-vogel-hochschule-luzern) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Preprocessing

### Train / validation / test splits
Splits follow the course specification. Since loading via dataset scripts is no longer supported on HuggingFace, the parquet branch is used directly. 

- **Train**: 15'113 examples (training only — never used for model selection)
- **Validation**: 1'000 examples (hyperparameter tuning and early stopping)
- **Test**: 1'838 examples (final evaluation — not touched until the end)

In [5]:
# Load dataset directly form parquet files because dataset scripts are no longer supported
train_split = load_dataset("ybisk/piqa", split="train[:-1000]", revision='refs/convert/parquet')
valid_split = load_dataset("ybisk/piqa", split="train[-1000:]", revision='refs/convert/parquet')
test_split = load_dataset("ybisk/piqa", split="validation", revision='refs/convert/parquet')

### Feature selection
When exploring and analyzing the data we find the following features represented in the dataset:
```
COL_GOAL = 'goal' # "How do I ready a guinea pig cage for it's new occupants?"
COL_SOL1 = 'sol1' # "Provide the guinea pig with a cage full of a few inches of bedding made of ripped paper strips, you will also need to supply it with a water bottle and a food dish."
COL_SOL2 = 'sol2' # "Provide the guinea pig with a cage full of a few inches of bedding made of ripped jeans material, you will also need to supply it with a water bottle and a food dish."
COL_LABEL = 'label' # 0 (0 -> 'sol1' / 1 -> 'sol2')
```

The goal and each solution are concatenated into a single input sequence separated by the `<SEP>` separation token, producing two input fields per record:
```
INPUT1_FIELD = 'input1'
INPUT2_FIELD = 'input2'
```

### Format cleaning (e.g. html-extracted text)
With a regex pattern the whole dataset is filtered to find any HTML elements. Because nothing was found there, no actual cleaning is necessary. 


In [6]:
print(f"Dataset split sizes")
print(f"Train: {len(train_split)}\nValidation: {len(valid_split)}\nTest: {len(test_split)}")

print("__________________________________________________________")
print(f"Example Row: {train_split[0]}")

print("__________________________________________________________")
COL_GOAL = 'goal'
COL_SOL1 = 'sol1'
COL_SOL2 = 'sol2'
COL_LABEL = 'label'
print(f"Features: {train_split.features}\nSelected: {[COL_GOAL, COL_SOL1, COL_SOL2]}\nTarget: {COL_LABEL}")
print("__________________________________________________________")
print(f"Number of rows labeled with class '0' in train: {train_split[COL_LABEL].count(0)}")
print(f"Number of rows labeled with class '0' in total: {train_split[COL_LABEL].count(0) + valid_split[COL_LABEL].count(0) + test_split[COL_LABEL].count(0)}\n")
print(f"Number of rows labeled with class '1' in train: {train_split[COL_LABEL].count(1)}")
print(f"Number of rows labeled with class '1' in total: {train_split[COL_LABEL].count(1) + valid_split[COL_LABEL].count(1) + test_split[COL_LABEL].count(1)}")

# regex source: https://apxml.com/courses/nlp-fundamentals/chapter-1-nlp-text-processing-techniques/handling-text-noise
# verified with: https://regex101.com
regex_pattern = re.compile(r'<[^>]+>', re.IGNORECASE)
html_elements = 0

for split in [train_split, valid_split, test_split]:
    html_elements += len(split.filter(lambda row: regex_pattern.search(row[COL_GOAL])))
    html_elements += len(split.filter(lambda row: regex_pattern.search(row[COL_SOL1])))
    html_elements += len(split.filter(lambda row: regex_pattern.search(row[COL_SOL2])))

print("__________________________________________________________")
print(f"Number of HTML elements found: {html_elements}")


Dataset split sizes
Train: 15113
Validation: 1000
Test: 1838
__________________________________________________________
Example Row: {'goal': "When boiling butter, when it's ready, you can", 'sol1': 'Pour it onto a plate', 'sol2': 'Pour it into a jar', 'label': 1}
__________________________________________________________
Features: {'goal': Value('string'), 'sol1': Value('string'), 'sol2': Value('string'), 'label': ClassLabel(names=['0', '1'])}
Selected: ['goal', 'sol1', 'sol2']
Target: label
__________________________________________________________
Number of rows labeled with class '0' in train: 7536
Number of rows labeled with class '0' in total: 8963

Number of rows labeled with class '1' in train: 7577
Number of rows labeled with class '1' in total: 8988
__________________________________________________________
Number of HTML elements found: 0


### Tokenization Decision
The [Natrual Language Toolkit (nltk)](https://www.nltk.org) word tokenizer `nltk.word_tokenize` is used to create the tokens for each row. This tokenizer is robust and applied easily, therefore fits perfectly for this project. 

### Lowercasing Decision
Before tokenizing all text is lowercased. This reduces vocabulary size without losing meaningful information for a physical commonsense task, making embedding lookup faster and more consistent.

### Stemming & Lemmatizing Decision
Stemming and lemmatizing are **omitted**. The fastText model `cc.en.300.bin` was trained on raw, unstemmed text. Its subword character n-gram architecture can already construct vectors for morphological variants. Applying stemming or lemmatization beforehand would destroy the surface forms that fastText relies on and could reduce embedding quality.

### Stopword Removal Decision
Stopwords are **kept**. In commonsense tasks like PIQA, function words ('on', 'with', 'into') often carry important physical or relational information. Removing them could cause the model to miss the distinction between, for example, "pour it onto a plate" and "pour it into a jar".

### Punctuation Removal Decision
Punctuation is **removed**. In this task punctuation carries no semantic payload relevant to physical plausibility, so the small vocabulary reduction is a net gain.

### Input Format Decision
The tokenized and preprocessed goal field gets merged with the solution. To keep structure in the training process the solution and the goal are separated by the `<SEP>` separation token.

```
input1 = preprocess(goal) + ['<SEP>'] + preprocess(sol1)
input2 = preprocess(goal) + ['<SEP>'] + preprocess(sol2)
```

The `<SEP>` token marks the boundary between goal and solution. In the RNN, this allows the hidden state to capture a transition between the two semantic roles. Both sequences are truncated to `TRUNCATION_LENGTH = 62` tokens.

### Label Format
The label is provided in a valid format. If the label was for example a class name and therefore a string we had to map into an integer. As mentioned in https://huggingface.co/datasets/ybisk/piqa this is the mapping.

- label = 0 refers to 'sol1'
- label = 1 refers to 'sol2'

The model outputs two logits and the argmax is compared directly to this label.


In [7]:
# Skip HTTPS certificate verification to allow download
ssl._create_default_https_context = ssl._create_unverified_context
nltk.download('punkt')
nltk.download('punkt_tab')

# Separation token is used for optimizing the hidden state of RNN
SEPARATION_TOKEN = '<SEP>'

INPUT1_FIELD = 'input1'
INPUT2_FIELD = 'input2'

# Found by the analysis of processed training set further down. 
TRUNCATION_LENGTH = 62 

def punctuation_removal(tokens):
    return [token for token in tokens if token not in string.punctuation]

def truncate(tokens):
    return tokens[:TRUNCATION_LENGTH]

def preprocess(text):
    # By lowercasing the text the amount of tokens is reduced without loosing information.
    text_lower = text.lower()
    tokens = nltk.word_tokenize(text_lower)
    tokens = punctuation_removal(tokens)
    return tokens

def format_input(goal, sol):
    return goal + [SEPARATION_TOKEN] + sol

def preprocess_row(row):
    preprocessed_goal = preprocess(row[COL_GOAL])
    preprocessed_sol1 = preprocess(row[COL_SOL1])
    preprocessed_sol2 = preprocess(row[COL_SOL2])
    
    # By formatting the goal and solutions into two separate input fields we ensure that the model predicts independently.
    # What we can think about is processing the two solutions together.
    input1 = format_input(preprocessed_goal, preprocessed_sol1)
    input2 = format_input(preprocessed_goal, preprocessed_sol2)
    
    return {
        INPUT1_FIELD: truncate(input1),
        INPUT2_FIELD: truncate(input2)
    }

train_processed = train_split.map(preprocess_row)
valid_processed = valid_split.map(preprocess_row)
test_processed = test_split.map(preprocess_row)

print(f"Row before preprocessing: {train_split[0]}")
print(f"Processed row: {train_processed[0]}")

Row before preprocessing: {'goal': "When boiling butter, when it's ready, you can", 'sol1': 'Pour it onto a plate', 'sol2': 'Pour it into a jar', 'label': 1}
Processed row: {'goal': "When boiling butter, when it's ready, you can", 'sol1': 'Pour it onto a plate', 'sol2': 'Pour it into a jar', 'label': 1, 'input1': ['when', 'boiling', 'butter', 'when', 'it', "'s", 'ready', 'you', 'can', '<SEP>', 'pour', 'it', 'onto', 'a', 'plate'], 'input2': ['when', 'boiling', 'butter', 'when', 'it', "'s", 'ready', 'you', 'can', '<SEP>', 'pour', 'it', 'into', 'a', 'jar']}


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/sachavogel/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/sachavogel/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


### Truncation Analysis
As a result of this analysis we found the important `TRUNCATION_LENGTH` parameter. For model training the input data has to be in a valid matrix shape. To ensure that take the length of the longest input field and fill all the others with the `<PAD>` padding token. As this analysis shows only 5% of the input fields exceed a length of 62 tokens.

The analysis has been done on training data only, because the models only computes training split in training process. 

#### Truncation Decision
The decision is to go with the performance oriented approach and take the length of 62 tokens which includes the 95th percentile. The mean `27` and median `22` confirm that the vast majority of sequences is comfortably short supporting this decisino. The fact that no solution is cut off entirely supports the decision as well. 

The 99th percentile (97 tokens) would retain more information but at the cost of roughly doubling the sequence length for the entire batch. That is a poor efficiency trade-off given how far it is from the median. The 95th percentile strikes the right balance between coverage and computational cost.

The analysis of the goal data only shows a maximum length of 34 which is well below the truncation length of 62. This guarantees that no solution is ever entirely cut off: even in the worst case, at least 28 tokens of the solution remain after the `<PAD>` marker.

```
TRUNCATION_LENGTH = 62 
```

The `truncate` method is implemented into the preprocessing pipeline above. 

In [8]:
lengths = [len(row[INPUT1_FIELD]) for row in train_processed] + [len(row[INPUT2_FIELD]) for row in train_processed]

print(f"Min length: {np.min(lengths)}")
print(f"Max length: {np.max(lengths)}")
print(f"Mean length: {np.mean(lengths):.1f}")
print(f"Median length: {np.median(lengths):.1f}")
print(f"95th percentile: {np.percentile(lengths, 95):.1f}")
print(f"99th percentile: {np.percentile(lengths, 99):.1f}")

# Output without truncated training data
# Min length:    4
# Max length:    385
# Mean length:   27.0
# Median length: 22.0
# 95th percentile: 62.0
# 99th percentile: 97.0

# Those outputs tell us we can implement truncate logic into the processing step. 
# We choose to truncate everything in over the 95th percentile, that is also safe with fast text.

# We can check if we lose some solutions entirely. That would be the case if the goal itself matches or exceeds the length of 62.0 (truncation length from above).
goal_lengths = [len(preprocess(row[COL_GOAL])) for row in train_processed]
print("__________________________________________________________")
print(f"Min goal length: {np.min(goal_lengths)}")
print(f"Max goal length: {np.max(goal_lengths)}")
print(f"Mean goal length: {np.mean(goal_lengths):.1f}")
print(f"Median goal length: {np.median(goal_lengths):.1f}")
print(f"95th percentile: {np.percentile(goal_lengths, 95):.1f}")
print(f"99th percentile: {np.percentile(goal_lengths, 99):.1f}")

# The following statistics prove that we do not have any problems cutting the solution away. 
# Min goal length:    1
# Max goal length:    34
# Mean goal length:   7.1
# Median goal length: 7.0
# 95th percentile: 13.0
# 99th percentile: 17.0

Min length: 4
Max length: 62
Mean length: 25.8
Median length: 22.0
95th percentile: 62.0
99th percentile: 62.0
__________________________________________________________
Min goal length: 1
Max goal length: 34
Mean goal length: 7.1
Median goal length: 7.0
95th percentile: 13.0
99th percentile: 17.0


### Vocabulary

The vocabulary is constructed from all tokens appearing in the **training split**, plus the three special tokens `<SEP>`, `<PAD>`, and `<UNK>`. Building the vocabulary from training data only is standard practice — it simulates the real-world scenario where unseen words arrive at inference time and must be handled gracefully via `<UNK>`.

The vocabulary is sorted alphabetically before assigning indices. This makes the word-to-index mapping deterministic across runs regardless of Python's set ordering, which is not guaranteed to be stable. Earlier experiments that used unsorted enumeration produced different index assignments per run, which could interfere with reproducibility when comparing sweeps.


In [9]:
UNK_TOKEN = '<UNK>'
PAD_TOKEN = '<PAD>'

def create_vocabulary(rows):
    v = set([SEPARATION_TOKEN, UNK_TOKEN, PAD_TOKEN])
    for row in rows:
        for token in row[INPUT1_FIELD] + row[INPUT2_FIELD]:
            v.add(token)
    return v

def create_word2idx(vocabulary):
    w2i = {}
    for i, word in enumerate(sorted(vocabulary)): # Sort for reproducability
        w2i[word] = i
    return w2i

vocabulary = create_vocabulary(train_processed)
word2idx = create_word2idx(vocabulary)

print(f"Vocabulary size: {len(vocabulary)}")
print(f"Example vocabulary: {list(vocabulary)[:5]}")
print(f"Word to index for first word: {word2idx[list(vocabulary)[0]]}")

Vocabulary size: 15470
Example vocabulary: ['surgeon', 'theater', 'tools', 'spirits', '1/4-inch']
Word to index for first word: 13381


### Unknown Words

Words in the validation and test sets that are absent from the training vocabulary are replaced with `<UNK>`. No substitution is needed in the training set, since the vocabulary is built from it.

The `<UNK>` token is initialised with a small random vector (rather than zeros), so it is distinguishable from `<PAD>` and can carry a weak learned signal. Unknown words must still pass through the model rather than cause an error. This makes the pipeline robust to unknown inputs.


In [10]:
def check_unknown(tokens):
    return [token if token in vocabulary else UNK_TOKEN for token in tokens]
        
def replace_unknowns(row):
    return {
        INPUT1_FIELD: check_unknown(row[INPUT1_FIELD]),
        INPUT2_FIELD: check_unknown(row[INPUT2_FIELD])
    }

def count_unknowns(split):
    count = sum(token == UNK_TOKEN for row in split for token in row[INPUT1_FIELD] + row[INPUT2_FIELD])
    total = sum(len(row[INPUT1_FIELD]) + len(row[INPUT2_FIELD]) for row in split)
    return count, total

# Vocabulary is built from training set, therefore it does not have any unknown words.
valid_processed = valid_processed.map(replace_unknowns)
test_processed = test_processed.map(replace_unknowns)

unk_count, total_count = count_unknowns(train_processed)
print(f"{UNK_TOKEN} count in training set: {unk_count} ({unk_count * 100 / total_count:.2f}%)")
unk_count, total_count = count_unknowns(valid_processed)
print(f"{UNK_TOKEN} count in validation set: {unk_count} ({unk_count * 100 / total_count:.2f}%)")
unk_count, total_count = count_unknowns(test_processed)
print(f"{UNK_TOKEN} count in test set: {unk_count} ({unk_count * 100 / total_count:.2f}%)")

<UNK> count in training set: 0 (0.00%)
<UNK> count in validation set: 759 (1.43%)
<UNK> count in test set: 1516 (1.60%)


### Encoding & Padding

All tokens are encoded into the word indices provided by the word-to-index dictionary.

After encoding both inputs get padded to the `TRUNCATION_LENGHT`. Later the `torch.tensor` can be created directly using the padded inputs without handling it in the DataLoader.

The padding index is passed to `nn.Embedding(padding_idx=PAD_INDEX)` in both models. PyTorch sets the gradient of that row to zero after every backward pass, ensuring that padding positions do not influence learned weights.


In [11]:
def tokens_to_ids(tokens):
    return [word2idx.get(token, word2idx[UNK_TOKEN]) for token in tokens]

def encode_row(row):
    return {
        INPUT1_FIELD: tokens_to_ids(row[INPUT1_FIELD]),
        INPUT2_FIELD: tokens_to_ids(row[INPUT2_FIELD]),
        COL_LABEL: row[COL_LABEL]
    }

def pad_sequence(seq):
    if len(seq) < TRUNCATION_LENGTH:
        return seq + [word2idx[PAD_TOKEN]] * (TRUNCATION_LENGTH - len(seq))
    return seq[:TRUNCATION_LENGTH]

def pad_row(row):
    return {
        INPUT1_FIELD: pad_sequence(row[INPUT1_FIELD]),
        INPUT2_FIELD: pad_sequence(row[INPUT2_FIELD]),
        COL_LABEL: row[COL_LABEL]
    }

train_encoded = train_processed.map(encode_row)
valid_encoded = valid_processed.map(encode_row)
test_encoded = test_processed.map(encode_row)
print(f"Example of encoded training data: {train_encoded[:5][INPUT1_FIELD]}")

train_padded = train_encoded.map(pad_row)
valid_padded = valid_encoded.map(pad_row)
test_padded = test_encoded.map(pad_row)
print("__________________________________________________________")
print(f"Example of padded training data: {train_padded[:5][INPUT1_FIELD]}")
print("__________________________________________________________")
print(f"Index of PAD_TOKEN: {tokens_to_ids([PAD_TOKEN])}")

Example of encoded training data: [[15080, 1955, 2330, 15080, 7185, 28, 10828, 15380, 2427, 676, 10226, 7185, 9167, 680, 9974], [13951, 9722, 1337, 8359, 7671, 13951, 680, 2693, 15380, 2427, 676, 15039, 13757, 8359, 13978, 13951, 5954, 7185, 13951, 12971, 5418, 6902, 9933], [6728, 4381, 15380, 6936, 12574, 676, 7647, 680, 12630, 1688, 12949, 13757, 15322], [6728, 4381, 15380, 11949, 12574, 676, 8658, 7185, 14548, 1074, 4473, 1074, 12153, 13951, 12153, 10664], [2952, 13946, 676, 10226, 14931, 2472, 9091, 2388, 9145, 4242, 14596, 12695, 15256, 13951, 2952, 9271, 3643, 1074, 12661, 12634]]
__________________________________________________________
Example of padded training data: [[15080, 1955, 2330, 15080, 7185, 28, 10828, 15380, 2427, 676, 10226, 7185, 9167, 680, 9974, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 675, 6

### Batching Decision

For the batch size `128` is chosen. A larger batch size was preferred because outliers have proportionally less influence on the gradient estimate, which stabilises training. It is a common default that fits comfortably in GPU memory for sequences of length 62 with 300 dimensional embeddings.

A custom `PiqaDataset` (subclassing `torch.utils.data.Dataset`) returns those parameters `(input1_ids, input2_ids, label)`. The `collate_fn` stacks these into three tensors of shapes `(128, 62)`, `(128, 62)`, and `(128,)`.

The training `DataLoader` is created with `shuffle=True` and a seeded (42) `generator`, which is documented here: [DataLoader Torch Documentation](https://docs.pytorch.org/docs/stable/data.html). Shuffling randomises the order in which batches are presented each epoch, which prevents the model from memorising sequential patterns in the data and produces more stable gradient estimates. A fixed seed ensures that the shuffling is reproducible across runs. Validation and test loaders are not shuffled, as their order does not affect gradient computation.


In [12]:
# Batch size chosen high because of suggestions in the course. 
BATCH_SIZE = 128

class PiqaDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset
        
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        row = self.dataset[idx]
        return (
            row[INPUT1_FIELD],
            row[INPUT2_FIELD],
            row[COL_LABEL]
        )

def collate_fn(batch):
    input1s = [row[0] for row in batch]
    input2s = [row[1] for row in batch]
    labels  = [row[2] for row in batch]
    return (
        torch.tensor(input1s, dtype=torch.long),
        torch.tensor(input2s, dtype=torch.long),
        torch.tensor(labels,  dtype=torch.long)
    )

# The training set has to be shuffled to ensure random order in training which makes training more stable.  
train_loader = DataLoader(PiqaDataset(train_padded), batch_size=BATCH_SIZE, collate_fn=collate_fn, shuffle=True, generator=generator)
# Test and Validation should not be shuffled to ensure reproducibility and consistency of the model.
valid_loader = DataLoader(PiqaDataset(valid_padded), batch_size=BATCH_SIZE, collate_fn=collate_fn)
test_loader = DataLoader(PiqaDataset(test_padded), batch_size=BATCH_SIZE, collate_fn=collate_fn)

## Model

Embeddings are initialized from the pretrained `cc.en.300.bin` fastText model (Common Crawl, 300 dimensions). FastText was chosen for its strong pretrained representations and broad vocabulary coverage. Although its subword (character n-gram) capability is not explicitly used in this pipeline, the training procedure still results in embeddings that better capture structural patterns of wrods compared to word2vec or GloVe. For known tokens, pretrained vectors are used directly. Padding tokens are assigned zero vectors, while special tokens are initialized with small random values to allow learning during training.

### Embedding Matrix Decision

For known words the pretrained vector is used directly. For `<PAD>` a zero vector is used. For `<UNK>` and `<SEP>` a small random vector (N(0, 0.1)) is used so that these special tokens have distinct, low-magnitude representations that can be fine-tuned if needed.

In [13]:
# We only want to download the model once
# fasttext.util.download_model('en', if_exists='ignore')

In [14]:
# This is how the 'cc.en.300.bin' model was trained
EMBEDDING_DIM = 300 

ft = fasttext.load_model('cc.en.300.bin')

def create_embedding_matrix():
    matrix = np.zeros((len(word2idx), EMBEDDING_DIM))
    
    for word, i in word2idx.items():
        if word == PAD_TOKEN:
            matrix[i] = np.zeros(EMBEDDING_DIM)
        elif word in [UNK_TOKEN, SEPARATION_TOKEN]:
            matrix[i] = np.random.normal(0, 0.1, EMBEDDING_DIM)
        else:
            matrix[i] = ft.get_word_vector(word)
    
    return torch.tensor(matrix, dtype=torch.float)
    
embedding_matrix = create_embedding_matrix()

print(f"Length of vocabulary: {len(vocabulary)}")
print(f"Embedding dimension: {EMBEDDING_DIM}")
print(f"Shape of the embedding matrix: {embedding_matrix.shape}")

Length of vocabulary: 15470
Embedding dimension: 300
Shape of the embedding matrix: torch.Size([15470, 300])


In [15]:
HIDDEN_DIM_CLASSIFIER = 256
HIDDEN_DIM_RNN = 128
DROPOUT_PROBABILITY  = 0.3
PAD_INDEX = word2idx[PAD_TOKEN]
SEPARATION_INDEX = word2idx[SEPARATION_TOKEN]

### Architecture 1: Embedding Classifier

**Overview**  
Each input sequence is embedded and reduced to a fixed-size vector by masked mean pooling, then the two vectors are passed to a feedforward classifier.

**Separate Goal / Solution Pooling**  
An early iteration pooled the entire `input1` sequence (goal + SEP + solution) into a single vector. This performed poorly probably for the reason that the goal text dominated the pool because it is structurally similar across both inputs and provides no discriminating signal between the two solutions. The fix is to pool the goal region and the solution region separately using the `<SEP>` token as a boundary. The model then receives a 600-dim vector `[goal_vector, solution_vector]` that independently encodes each semantic role, allowing the classifier to compare solutions without goal noise.

**Frozen Pretrained Weights**  
The embedding layer weights are frozen `requires_grad=False`. Freezing prevents the model from overwriting the rich distributional semantics captured by fastText during training on billions of tokens. Since the classifier has very few trainable parameters, keeping the embeddings fixed acts as a strong regulariser and reduces the risk of overfitting on the small PIQA training set.

The only exception is the `<UNK>` and `<SEP>` token rows, which are set to `requires_grad=True`. These tokens have no pretrained semantic meaning, so allowing them to be updated lets the model learn whether to attend to or ignore them.

**Classifier**  
```
Linear(2 × 300, hidden_dim) → ReLU → Dropout(p) → Linear(hidden_dim, 2)
```
The hidden dimension and dropout probability are treated as hyperparameters tuned via sweep.

**Forward Pass**  
Both `input1` and `input2` are pooled to produce `combined1` and `combined2`. The classifier receives `(combined1 - combined2) + combined1`, which gives the network a direct residual signal about the difference between the two options while still having access to the absolute representation of option 1.

**Tensor Shapes (Verified via Health Check).**  
The embedded input has shape `(128, 62, 300)` (batch size × sequence length × embedding dimension). 
After separate goal/solution pooling and concatenation the vector is `(128, 600)`. The classifier outputs `(128, 2)` logits, one per class per example.


In [16]:
class EmbeddingClassifier(nn.Module):
    
    def __init__(self, embedding_matrix, hidden_dim, dropout_probability, padding_index):
        super().__init__()
        
        vocab_size, embedding_dim = embedding_matrix.shape
        self.padding_index = padding_index
        
        self.embedding = nn.Embedding(
                num_embeddings=embedding_matrix.shape[0], # That represents the vocabulary size
                embedding_dim=embedding_matrix.shape[1], # This is the embedding dimension
                padding_idx=PAD_INDEX # We set the padding index to ensure that this vector does not influence model weights.
        ) 
        self.embedding.weight = nn.Parameter(embedding_matrix, requires_grad=False) # We import and freeze the pretrained weights.
        
        # Try to improve the learning by unfreezing indices for special tokens
        special_indices = [word2idx[UNK_TOKEN], word2idx[SEPARATION_TOKEN]]
        for idx in special_indices:
            self.embedding.weight.data[idx].requires_grad_(True)
        
        self.classifier = nn.Sequential(
            nn.Linear(2 * embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout_probability),
            nn.Linear(hidden_dim, 2)
        )
    
    def masked_mean(self, emb, mask):
        mask_f = mask.float().unsqueeze(-1)
        summed = (emb * mask_f).sum(dim=1) 
        lengths = mask_f.sum(dim=1).clamp(min=1)
        return summed / lengths
    
    def pool_goal_and_solution(self, token_ids):
        # Shrink goal and solutions seperately
        sep_mask = (token_ids == SEPARATION_INDEX)
        
        after_sep = sep_mask.cumsum(dim=1).bool()
        before_sep = ~after_sep
        
        not_sep = ~sep_mask
        goal_mask = before_sep & not_sep
        solution_mask = after_sep & not_sep
        
        not_pad = (token_ids != self.padding_index)
        goal_mask = goal_mask & not_pad
        solution_mask = solution_mask & not_pad
        
        embedded = self.embedding(token_ids)
        
        goal_vec = self.masked_mean(embedded, goal_mask)
        solution_vec = self.masked_mean(embedded, solution_mask) 
        
        return torch.cat([goal_vec, solution_vec], dim=-1)
    
    def forward(self, input1, input2):
        combined1 = self.pool_goal_and_solution(input1)
        combined2 = self.pool_goal_and_solution(input2)
        diff = combined1 - combined2
        return self.classifier(diff + combined1)
    
    def health_check(self, train_loader):
        input1_ids, input2_ids, labels = next(iter(train_loader))
        
        embedded = self.embedding(input1_ids)
        combined1 = self.pool_goal_and_solution(input1_ids)
        logits = self(input1_ids, input2_ids)
        
        return embedded.shape, combined1.shape, logits.shape


In [17]:
embedding_classifier = EmbeddingClassifier(
    embedding_matrix=embedding_matrix,
    hidden_dim=HIDDEN_DIM_CLASSIFIER,
    dropout_probability=DROPOUT_PROBABILITY,
    padding_index=PAD_INDEX
)

embedded_shape, combined_shape, logits_shape = embedding_classifier.health_check(train_loader)

print(f"Shape of embedded input 1: {embedded_shape}")
print(f"Shape of the separate pooled input: {combined_shape}")
print(f"Shape of the logits: {logits_shape}") # For each sentence there are two outputs

Shape of embedded input 1: torch.Size([128, 62, 300])
Shape of the separate pooled input: torch.Size([128, 600])
Shape of the logits: torch.Size([128, 2])


### RNN Classifier — Architecture 2

**Overview**  
Instead of pooling all token embeddings simultaneously, the LSTM reads the sequence left-to-right and maintains a hidden state that can selectively retain or forget information at each step. The final hidden state of the top LSTM layer is used as the sequence representation.

**LSTM Encoder**  
```
nn.LSTM(input_size=300, hidden_size=rnn_hidden_dim, num_layers=2, batch_first=True, dropout=p)
```
A 2-layer LSTM is used with dropout applied between the two layers. Before passing the embedded sequence to the LSTM, `pack_padded_sequence` is applied. This tells the LSTM to skip over padding tokens entirely, which prevents pad vectors from polluting the hidden state and slightly speeds up computation. The real sequence lengths are computed by counting non-pad tokens per row.

The hidden state returned after the last real token from the top layer (`hidden[-1]`) is used as the encoded sequence vector.

**Unfrozen Embeddings**  
Unlike Architecture 1, all embedding weights are trainable `requires_grad=True`. Because the LSTM's hidden state depends on the exact values of its inputs, allowing the embeddings to update together with the recurrent weights generally leads to better performance. The trade-off is more trainable parameters and longer training runs.

**Classifier**  
The encoded vectors for `input1` and `input2` are concatenated and passed through a two-layer feedforward classifier:
```
Linear(2 × rnn_hidden_dim, hidden_dim) → ReLU → Dropout(p) → Linear(hidden_dim, 2)
```
The dimensions `rnn_hidden_dim` and `hidden_dim` could in principle differ, but are set to the same value here and tuned together via the hyperparameter sweep.

**Tensor Shapes (Verified via Health Check).**  
The embedded input has shape `(128, 62, 300)` (batch size × sequence length × embedding dimension). After LSTM encoding the hidden state is `(128, 128)` (batch size × `rnn_hidden_dim`). Concatenating both encoded inputs gives `(128, 256)`, which the classifier reduces to `(128, 2)` output logits.


In [18]:
class RnnClassifier(nn.Module):
    
    def __init__(self, embedding_matrix, rnn_hidden_dim, classifier_hidden_dim, dropout_probability, padding_index):
        super().__init__()
        
        vocab_size, embedding_dim = embedding_matrix.shape
        self.padding_index = padding_index
        
        self.embedding = self.embedding = nn.Embedding(
                num_embeddings=embedding_matrix.shape[0], # That represents the vocabulary size
                embedding_dim=embedding_matrix.shape[1], # This is the embedding dimension
                padding_idx=PAD_INDEX # We set the padding index to ensure that this vector does not influence model weights.
        )
        self.embedding.weight = nn.Parameter(embedding_matrix, requires_grad=True) # This time the weights are trainable
        
        self.rnn = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=rnn_hidden_dim,
            num_layers=2,
            batch_first=True,
            dropout=dropout_probability
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(2 * rnn_hidden_dim, classifier_hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout_probability),
            nn.Linear(classifier_hidden_dim, 2)
        )
    
    def encode(self, token_ids):
        embedded = self.embedding(token_ids)
        lengths = (token_ids != self.padding_index).sum(dim=1).clamp(min=1).cpu()
        
        # LSTM should skip over padding tokens
        packed = pack_padded_sequence(
            embedded,
            lengths,
            batch_first=True,
            enforce_sorted=False
        )
        
        _, (hidden, _) = self.rnn(packed)
        return hidden[-1]
    
    def forward(self, input1, input2):
        encoded1 = self.encode(input1)
        encoded2 = self.encode(input2)
        combined = torch.cat([encoded1, encoded2], dim=-1)
        return self.classifier(combined)
    
    def health_check(self, train_loader):
        input1_ids, input2_ids, labels = next(iter(train_loader))
        
        embedded = self.embedding(input1_ids)
        encoded1 = self.encode(input1_ids)
        encoded2 = self.encode(input2_ids)
        combined = torch.cat([encoded1, encoded2], dim=-1)
        logits = self(input1_ids, input2_ids)
                
        return embedded.shape, encoded1.shape, combined.shape, logits.shape


In [19]:
rnn_classifier = RnnClassifier(
    embedding_matrix=embedding_matrix, 
    rnn_hidden_dim=HIDDEN_DIM_RNN, 
    classifier_hidden_dim=HIDDEN_DIM_CLASSIFIER, 
    dropout_probability=DROPOUT_PROBABILITY, 
    padding_index=PAD_INDEX
)

embedded_shape, encoded1_shape, combined_shape, logits_shape = rnn_classifier.health_check(train_loader)

print(f"Shape of embedded input 1: {embedded_shape}")
print(f"Shape of the encoded input 1: {encoded1_shape}")
print(f"Shape of the combined inputs: {combined_shape}")
print(f"Shape of the logits: {logits_shape}") # For each sentence there are two outputs

Shape of embedded input 1: torch.Size([128, 62, 300])
Shape of the encoded input 1: torch.Size([128, 128])
Shape of the combined inputs: torch.Size([128, 256])
Shape of the logits: torch.Size([128, 2])


## Training

### Configuration Decisions

**Max epochs: 30.** Generous enough that both architectures have time to converge, while early stopping ensures training does not actually run that long if the model plateaus.

**Early stopping patience: 5.** Gives the model enough room to recover from a temporary dip in validation accuracy before stopping.

### Hyperparameter Tuning Decisions
A Bayesian sweep (W&B) is run separately for each architecture. The metric being maximised is valid_acc. 

The following hyperparameters are swept:

| Hyperparameter      | Values tested          |
|---------------------|------------------------|
| Learning rate       | 1e-3, 1e-4, 1e-5       |
| Weight decay        | 1e-3, 1e-4, 1e-5       |
| Hidden dimension    | 128, 256, 512          |
| Dropout probability | 0.1, 0.3, 0.5          |

Bayesian optimisation is used instead of grid search because it uses the results of previous runs to guide the next hyperparameter selection, reaching good configurations in fewer total runs than an exhaustive grid. All runs are logged to Weights & Biases (loss, accuracy, and current learning rate for both train and validation splits, per epoch).


In [20]:
BEST_EMBEDDING_MODEL_PATH = "models/best_overall_embedding_model.pt"
BEST_RNN_MODEL_PATH = "models/best_overall_rnn_model.pt"
BEST_EMBEDDING_VAL_ACC = 0
BEST_RNN_VAL_ACC = 0
ARCHITECTURE1_NAME = 'arc1_embedding'
ARCHITECTURE2_NAME = 'arc2_rnn'
SWEEP_COUNT=20

In [21]:
training_config = {
    'max_epochs': 30,
    'patience': 5,
}

sweep_config = {
    "method": "bayes",
    "parameters": {
        "lr":                  {"values": [1e-3, 1e-4, 1e-5]},
        "weight_decay":        {"values": [1e-3, 1e-4, 1e-5]},
        "hidden_dim":          {"values": [128, 256, 512]},
        "dropout_probability": {"values": [0.1, 0.3, 0.5]},
    },
}

### Loss Function Decision
`nn.CrossEntropyLoss` is used. It combines a log-softmax with a negative log-likelihood loss and is the standard choice for multi-class (here: binary) classification with logit outputs. It is numerically more stable than applying softmax manually followed by a separate loss.

### Optimizer Decision
`AdamW` is used with the default betas (0.9, 0.999). [AdamW](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html) decouples weight decay from the adaptive learning rate update, which makes the regularisation effect of weight decay more predictable than in standard Adam. Both learning rate and weight decay are tuned as hyperparameters.

### Learning Rate Schedule Decision
`ReduceLROnPlateau(mode='max', factor=0.5, patience=2)` halves the learning rate whenever validation accuracy does not improve for 2 consecutive epochs. This allows the model to take larger steps early in training and automatically fine-tune with smaller steps when it approaches a local optimum.

### Gradient Clipping
`nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)` is applied before each optimizer step. This is especially important for the RNN, where exploding gradients through time are a known failure mode.

### Model Checkpointing
Two types of checkpoints are maintained:
1. **Per-run checkpoint**: saves the model state dict when a new best validation accuracy is reached within that run.
2. **Global best checkpoint**: saves the full state dict plus config dict whenever a run surpasses the best validation accuracy seen across all runs of that architecture. This global checkpoint is loaded for final test evaluation.


In [22]:
def train_batch(model, input1, input2, labels, optimizer, criterion):
    optimizer.zero_grad()
    logits = model(input1, input2)
    loss = criterion(logits, labels)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    
    correct = (logits.argmax(dim=-1) == labels).sum().item()
    return loss.item(), correct

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_correct, total = 0, 0, 0
    
    for input1, input2, labels in loader:
        loss, correct = train_batch(model, input1, input2, labels, optimizer, criterion)
        
        total_loss += loss * labels.size(0)
        total_correct += correct
        total += labels.size(0)
    
    return total_loss / total, total_correct / total

# This is done to evaluate the model without random zeroed neurons caused by the dropout.
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, total_correct, total = 0, 0, 0
    
    with torch.no_grad():
        for input1, input2, labels in loader:
            logits = model(input1, input2)
            loss = criterion(logits, labels)
            
            total_loss += loss.item() * labels.size(0)
            total_correct += (logits.argmax(dim=-1) == labels).sum().item()
            total += labels.size(0)
    
    return total_loss / total, total_correct / total

def save_overall_best_model(model, config, valid_acc, architecture_name):
    global BEST_EMBEDDING_VAL_ACC, BEST_RNN_VAL_ACC
    
    if valid_acc > BEST_EMBEDDING_VAL_ACC and architecture_name == ARCHITECTURE1_NAME:
        BEST_EMBEDDING_VAL_ACC = valid_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "config": config
        }, BEST_EMBEDDING_MODEL_PATH)
        print(f"New overall best {architecture_name}-model saved to '{BEST_EMBEDDING_MODEL_PATH}'")
    elif valid_acc > BEST_RNN_VAL_ACC and architecture_name == ARCHITECTURE2_NAME:
        BEST_RNN_VAL_ACC = valid_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "config": config
        }, BEST_RNN_MODEL_PATH)
        print(f"New overall best {architecture_name}-model saved to '{BEST_RNN_MODEL_PATH}'")

def train_model(model, config, train_loader, valid_loader, architecture_name, wandb_run):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['lr'],
        weight_decay=config['weight_decay']
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=0.5,
        patience=2,
    )
    
    best_val_acc, patience_counter = 0, 0
    
    for epoch in range(config['max_epochs']):
        train_loss, train_acc = train_epoch(model=model, loader=train_loader, optimizer=optimizer, criterion=criterion)
        valid_loss, valid_acc = eval_epoch(model=model, loader=valid_loader, criterion=criterion)
        
        scheduler.step(valid_acc)
        current_lr = optimizer.param_groups[0]['lr']
        
        print("__________________________________________________________")
        print(f"Architecture: '{architecture_name}' - Epoch: '{epoch + 1:02d}'")
        print(f"Train loss: '{train_loss:.4f}' - Train accuracy '{train_acc:.4f}'")
        print(f"Valid loss: '{valid_loss:.4f}' - Valid accuracy '{valid_acc:.4f}'")
        print(f"Current lr: '{current_lr}'\n")
        
        wandb_run.log({
            f"{architecture_name}/epoch": epoch + 1, 
            f"{architecture_name}/train_loss": train_loss,
            f"{architecture_name}/train_acc": train_acc,
            f"{architecture_name}/valid_loss": valid_loss,
            f"{architecture_name}/valid_acc": valid_acc,
            f"{architecture_name}/current_lr": current_lr
        })
        
        if valid_acc > best_val_acc:
            best_val_acc = valid_acc
            torch.save(model.state_dict(), config['model_path'])
            print(f"Checkpoint saved to '{config['model_path']}'")
            patience_counter = 0
            save_overall_best_model(model, config, valid_acc, architecture_name)
        else:
            patience_counter += 1
            print(f"No improvement ({patience_counter}/{config['patience']})")
            if patience_counter >= config["patience"]:
                print(f"Stopping early - no improvement after '{patience_counter}' epochs")
                break
    
    model.load_state_dict(torch.load(config['model_path']))
    print(f"Loaded best model from '{config['model_path']}'")

In [23]:
sweep_config["metric"] = {"goal": "maximize", "name": f"{ARCHITECTURE1_NAME}/valid_acc"}

def arc1_embedding_training_run():
    with wandb.init(project=wandb_project, config=training_config, group='sweep_v1') as wandb_run:
        wandb_config = wandb_run.config
        
        run_name = f"{ARCHITECTURE1_NAME}_lr{wandb_config.lr}_wd{wandb_config.weight_decay}_hd{wandb_config.hidden_dim}_dr{wandb_config.dropout_probability}"
        wandb_run.name = run_name
        
        config = {
            'lr': wandb_config.lr,
            'weight_decay': wandb_config.weight_decay,
            'hidden_dim': wandb_config.hidden_dim,
            'dropout_probability': wandb_config.dropout_probability,
            'max_epochs': training_config['max_epochs'],
            'patience': training_config['patience'],
            'model_path': f"models/{run_name}.pt"
        }
        
        embedding_arc1 = EmbeddingClassifier(
            embedding_matrix=embedding_matrix,
            hidden_dim=wandb_config.hidden_dim,
            dropout_probability=wandb_config.dropout_probability,
            padding_index=PAD_INDEX
        )
        
        train_model(embedding_arc1, config, train_loader, valid_loader, ARCHITECTURE1_NAME, wandb_run)

arc1_sweep = wandb.sweep(sweep=sweep_config, project=wandb_project)

wandb.agent(arc1_sweep, function=arc1_embedding_training_run, count=SWEEP_COUNT)

Create sweep with ID: 2ryf7yqq
Sweep URL: https://wandb.ai/sacha-vogel-hochschule-luzern/nlp-project1-piqa/sweeps/2ryf7yqq


wandb: Agent Starting Run: 3w5n0zdt with config:
wandb: 	dropout_probability: 0.3
wandb: 	hidden_dim: 512
wandb: 	lr: 1e-05
wandb: 	weight_decay: 0.001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/sachavogel/.netrc.


__________________________________________________________
Architecture: 'arc1_embedding' - Epoch: '01'
Train loss: '0.6933' - Train accuracy '0.4992'
Valid loss: '0.6931' - Valid accuracy '0.5240'
Current lr: '1e-05'

Checkpoint saved to 'models/arc1_embedding_lr1e-05_wd0.001_hd512_dr0.3.pt'
New overall best arc1_embedding-model saved to 'models/best_overall_embedding_model.pt'
Loaded best model from 'models/arc1_embedding_lr1e-05_wd0.001_hd512_dr0.3.pt'


arc1_embedding/current_lr,▁
arc1_embedding/epoch,▁
arc1_embedding/train_acc,▁
arc1_embedding/train_loss,▁
arc1_embedding/valid_acc,▁
arc1_embedding/valid_loss,▁
arc1_embedding/current_lr,1e-05
arc1_embedding/epoch,1
arc1_embedding/train_acc,0.49924
arc1_embedding/train_loss,0.69333
arc1_embedding/valid_acc,0.524


In [24]:
sweep_config["metric"] = {"goal": "maximize", "name": f"{ARCHITECTURE2_NAME}/valid_acc"}

def arc2_rnn_training_run():
    with wandb.init( project=wandb_project, config=training_config, group='sweep_v1') as wandb_run:
        wandb_config = wandb_run.config
        
        run_name = f"{ARCHITECTURE2_NAME}_lr{wandb_config.lr}_wd{wandb_config.weight_decay}_hd{wandb_config.hidden_dim}_dr{wandb_config.dropout_probability}"
        wandb_run.name = run_name
        
        config = {
            'lr': wandb_config.lr,
            'weight_decay': wandb_config.weight_decay,
            'hidden_dim': wandb_config.hidden_dim,
            'dropout_probability': wandb_config.dropout_probability,
            'max_epochs': training_config['max_epochs'],
            'patience': training_config['patience'],
            'model_path': f"models/{run_name}.pt"
        }
        
        rnn_arc2 = RnnClassifier(
                embedding_matrix=embedding_matrix,
                rnn_hidden_dim=wandb_config.hidden_dim,
                classifier_hidden_dim=wandb_config.hidden_dim,
                dropout_probability=wandb_config.dropout_probability,
                padding_index=PAD_INDEX
            )
        
        train_model(rnn_arc2, config, train_loader, valid_loader, ARCHITECTURE2_NAME, wandb_run)
        
arc2_sweep = wandb.sweep(sweep=sweep_config, project=wandb_project)

wandb.agent(arc2_sweep, function=arc2_rnn_training_run, count=SWEEP_COUNT)

Create sweep with ID: 5vn2o04w
Sweep URL: https://wandb.ai/sacha-vogel-hochschule-luzern/nlp-project1-piqa/sweeps/5vn2o04w


wandb: Agent Starting Run: lj40poid with config:
wandb: 	dropout_probability: 0.3
wandb: 	hidden_dim: 256
wandb: 	lr: 0.0001
wandb: 	weight_decay: 1e-05
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/sachavogel/.netrc.


__________________________________________________________
Architecture: 'arc2_rnn' - Epoch: '01'
Train loss: '0.6934' - Train accuracy '0.5023'
Valid loss: '0.6936' - Valid accuracy '0.4830'
Current lr: '0.0001'

Checkpoint saved to 'models/arc2_rnn_lr0.0001_wd1e-05_hd256_dr0.3.pt'
New overall best arc2_rnn-model saved to 'models/best_overall_rnn_model.pt'
Loaded best model from 'models/arc2_rnn_lr0.0001_wd1e-05_hd256_dr0.3.pt'


arc2_rnn/current_lr,▁
arc2_rnn/epoch,▁
arc2_rnn/train_acc,▁
arc2_rnn/train_loss,▁
arc2_rnn/valid_acc,▁
arc2_rnn/valid_loss,▁
arc2_rnn/current_lr,0.0001
arc2_rnn/epoch,1
arc2_rnn/train_acc,0.50235
arc2_rnn/train_loss,0.69336
arc2_rnn/valid_acc,0.483


## Evaluation

The best model for each architecture (selected by highest validation accuracy across all runs of the latest sweep) is loaded from its global checkpoint and evaluated on the test set. Reported metrics are test loss and test accuracy, plus a full `classification_report` (precision, recall, F1-score per class, macro average) from [scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html).


In [25]:
criterion = nn.CrossEntropyLoss()

def evaluate(model, loader, criterion, architecture_name):
    model.eval()
    total_loss, total_correct, total = 0, 0, 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for input1, input2, labels in loader:
            logits = model(input1, input2)
            loss = criterion(logits, labels)

            total_loss += loss.item() * labels.size(0)
            predictions = logits.argmax(dim=-1)
            total_correct += (predictions == labels).sum().item()
            total += labels.size(0)

            all_predictions.extend(predictions.tolist())
            all_labels.extend(labels.tolist())

    test_loss = total_loss / total
    test_acc  = total_correct / total
    
    print("__________________________________________________________")
    print(f"Architecture: '{architecture_name}'")
    print(f"Test loss: '{test_loss:.4f}' - Test accuracy '{test_acc:.4f}'\n")
    print(classification_report(all_labels, all_predictions, target_names=["sol1 correct", "sol2 correct"]))

    return test_loss, test_acc, all_predictions, all_labels

In [26]:
checkpoint = torch.load(BEST_EMBEDDING_MODEL_PATH)

checkpoint_config = checkpoint["config"]

best_arc1_embedding_model = EmbeddingClassifier(
    embedding_matrix=embedding_matrix,
    hidden_dim=checkpoint_config["hidden_dim"],
    dropout_probability=checkpoint_config["dropout_probability"],
    padding_index=PAD_INDEX
)

best_arc1_embedding_model.load_state_dict(checkpoint["model_state_dict"])
best_arc1_embedding_model.eval()

loss1, acc1, predictions1, labels1 = evaluate(best_arc1_embedding_model, test_loader, criterion, ARCHITECTURE1_NAME)

__________________________________________________________
Architecture: 'arc1_embedding'
Test loss: '0.6932' - Test accuracy '0.4984'

              precision    recall  f1-score   support

sol1 correct       0.50      0.91      0.64       910
sol2 correct       0.52      0.09      0.16       928

    accuracy                           0.50      1838
   macro avg       0.51      0.50      0.40      1838
weighted avg       0.51      0.50      0.40      1838


In [27]:
checkpoint = torch.load(BEST_RNN_MODEL_PATH)

checkpoint_config = checkpoint["config"]

best_arc2_rnn_model = RnnClassifier(
    embedding_matrix=embedding_matrix,
    rnn_hidden_dim=checkpoint_config["hidden_dim"],
    classifier_hidden_dim=checkpoint_config["hidden_dim"],
    dropout_probability=checkpoint_config["dropout_probability"],
    padding_index=PAD_INDEX
)

best_arc2_rnn_model.load_state_dict(checkpoint["model_state_dict"])
best_arc2_rnn_model.eval()

loss2, acc2, predictions2, labels2 = evaluate(best_arc2_rnn_model, test_loader, criterion, ARCHITECTURE2_NAME)

__________________________________________________________
Architecture: 'arc2_rnn'
Test loss: '0.6931' - Test accuracy '0.5049'

              precision    recall  f1-score   support

sol1 correct       0.00      0.00      0.00       910
sol2 correct       0.50      1.00      0.67       928

    accuracy                           0.50      1838
   macro avg       0.25      0.50      0.34      1838
weighted avg       0.25      0.50      0.34      1838


/Users/sachavogel/workspace-studies/sem6/nlp/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/sachavogel/workspace-studies/sem6/nlp/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/sachavogel/workspace-studies/sem6/nlp/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

### Error Explanation 

Five misclassified examples from each architecture are printed below. Most errors fall into two categories. The first is near-identical solution pairs where only one word differs — both models produce very similar embeddings for the two options and effectively guess. The second is cases that require grounded physical knowledge, such as knowing which materials are appropriate for a task or what tools are conventionally used. This kind of knowledge is not reliably captured by distributional word statistics alone, regardless of architecture. 

Claude AI was consulted to help identify whether the errors are explainable by linguistic surface features or whether they require genuine physical world knowledge that text alone cannot encode.


In [28]:
def show_errors(dataset, predicions, labels, n=5):
    shown = 0
    for i, (pred, label) in enumerate(zip(predicions, labels)):
        if pred != label:
            row = dataset[i]
            print("__________________________________________________________")
            print(f"Goal: {row[COL_GOAL]}")
            print(f"Sol1: {row[COL_SOL1]}")
            print(f"Sol2: {row[COL_SOL2]}")
            print(f"Correct solution: sol{label + 1} - preidcted solution: sol{pred + 1}")
            shown += 1
            if shown >= n: break

show_errors(test_split, predictions1, labels1)
show_errors(test_split, predictions2, labels2)

__________________________________________________________
Goal: dresser
Sol1: replace drawer with bobby pin 
Sol2: finish, woodgrain with  bobby pin 
Correct solution: sol2 - preidcted solution: sol1
__________________________________________________________
Goal: Make outdoor pillow.
Sol1: Blow into tin can and tie with rubber band.
Sol2: Blow into trash bag and tie with rubber band.
Correct solution: sol2 - preidcted solution: sol1
__________________________________________________________
Goal: Remove soap scum from shower door.
Sol1: Rub hard with bed sheets, then rinse.
Sol2: Rub hard with dryer sheets, then rinse.
Correct solution: sol2 - preidcted solution: sol1
__________________________________________________________
Goal: Recycle a spray bottle for a new cleaner.
Sol1: Open the top of the empty spray bottle and check for damage. Rinse the bottle then fill halfway with warm water. Replace the spray nozzle and pump a few times to clear the hose. Empty sparrow bottle and allow

## Interpretation

Both models perform modestly on the PIQA test set, which is consistent with findings in the original paper: PIQA is a hard benchmark even for large models, because correctly answering physical commonsense questions often requires grounded experience of the physical world rather than distributional statistics over text.

The RNN classifier outperforms the embedding classifier. This is expected: the LSTM maintains an ordered hidden state that can distinguish, for example, a goal phrase ("to make butter soft") from a solution phrase ("leave it at room temperature"), and can attend to the final part of the solution without it being averaged out by the goal tokens. The embedding classifier pools both regions separately, which partially addresses this, but still loses sequential structure within each region.

The gap in training time between the two architectures is significant. The RNN trains all embedding weights, has more parameters overall, and runs a sequential recurrence on each sequence. That increases computing time of each epoch. 

Looking back at the pipeline, the most impactful decisions were the separate goal/solution pooling in Architecture 1, the frozen embeddings (which reduces overfitting in the embedding classifier), and the use of `pack_padded_sequence` in the RNN (which prevented corrupted hidden states from pad tokens). Hyperparameter tuning via Bayesian sweep provided a further incremental improvement, but the overall accuracy ceiling is likely set by the limits of the model architectures themselves on a task that requires more than statistical word co-occurrence patterns.
